# Federal Election Simulation with Real Voting Data

This notebook demonstrates, using the 2021 German federal election as an example, how real voting data can be injected into the election simulation and used as a starting point to simulate various possible election outcomes.

## 1. Create Configuration and Load Voting Data for the First Round

In [ ]:
import numpy as np

from ipres import SuperMajorityMargin, SeatDistributionMethod, MarginUnit
from ipres import ElectionEvaluator, contestantsFromParties, Election, ElectionConfig, ConstituenciesConfig, VoteMatrix
from ipres.bundestagswahl_loader import load_bundestagswahl_data
from ipres.election_config import QuotaCorrectionStrategy, ConstituencyRepresentation, DrawLotsStrategy, Language

# Load federal election 2021 data
constituencies_df, vote_matrix, party_names = load_bundestagswahl_data(2021)
constituencies_config = ConstituenciesConfig(constituencies=constituencies_df)
contestants = contestantsFromParties(party_names)

# ElectionConfig with seed for reproducibility
election_config = ElectionConfig(
    constituencies_config=constituencies_config,
    participating_parties=party_names,
    parliament_majority_margin = SuperMajorityMargin(10, MarginUnit.SEATS),
    draw_lots_strategy = DrawLotsStrategy.RANDOM,
    language = Language.EN,
    seed=8685
    # seed = 8685  Union wins by drawing of lots with a required 10-seat governing-majority advantage and explicit random draw (parliament_majority_margin = SuperMajorityMargin(10, MarginUnit.SEATS), draw_lots_strategy = DrawLotsStrategy.RANDOM)
    # seed=8685    FDP wins with a required 10-seat governing-majority advantage using marginal lead instead of random draw (parliament_majority_margin = SuperMajorityMargin(10, MarginUnit.SEATS), draw_lots_strategy = DrawLotsStrategy.MARGINAL_LEAD)
    #seed=1169    #Union wins by drawing of lots with a 5% minimum difference (SuperMajorityMargin(5, MarginUnit.PERCENT))
    #seed=6       #Union wins directly in the 5th round with a 5% minimum difference (SuperMajorityMargin(5, MarginUnit.PERCENT))
)
# Create vote matrix from real voting data (vote_matrix injection)
vote_matrix_real = VoteMatrix.generate(
    constituencies_config=constituencies_config,
    contestants=contestants,
    vote_matrix=vote_matrix  # Inject real data!
)

print(f"Number of constituencies: {len(constituencies_config.getConstituencyNames())}")
print(f"Number of parties: {len(contestants)}")
print(f"Constituency representation: {'Entire parliament' if election_config.constituency_representation == ConstituencyRepresentation.ENTIRE_PARLIAMENT else 'Governing majority'}")
print(f"Parliamentary seats: {election_config.parliamentarySeats}")
print(f"Governing majority seats: {election_config.getParliamentMajoritySeats()}")
print(f"Governing majority: {round(election_config.getParliamentMajorityPercent(),2)} %")
print("First round from real 2021 German federal election data.")
print(f"\nTotal votes per party in the first round:")
print(vote_matrix_real.getVotes().sum(axis=0).sort_values(ascending=False))

## 2. Simulation Run

In [ ]:
# Election with real data as the first round
from ipres import ElectionRoundInput, ElectionRound, DrawLotsStrategy
from matplotlib import pyplot as plt
election = Election(electionConfig=election_config)

# Create ElectionRoundInput with real voting data
iteration_input = ElectionRoundInput(
    election=election,
    constituencies_config=constituencies_config,
    contestants={c.name: c for c in contestants},
    ballot_majority_percent=election_config.getParliamentMajorityPercent(),  # The same threshold as for the parliamentary majority is used here; a different ballot-round threshold could be configured via election_config.getBallotMajorityPercent(). (This would, however, produce different results.)
    draw_lots_strategy=election_config.draw_lots_strategy,
    rng = np.random.default_rng(election_config.seed),
    #rng=np.random.default_rng(1169), #Union wins by drawing of lots with a 5% minimum difference
    #rng=np.random.default_rng(6), #Union wins directly in the 5th round
    vote_matrix=vote_matrix_real  # Inject real federal election 2021 data
)

# Start first round with real data
iteration = election.start(iteration_input)
iteration.formCoalition("Union", ["CDU", "CSU"])
title = f"Round {iteration.getRoundNumber()} (real data from federal election 2021)"
fig = iteration.plot_vote_share_pie(title=title, min_percent = 2.0 )
display(fig)
plt.close(fig)

while iteration.hasNext():
    iterationInput = iteration.getNextRoundInput()
    iteration = ElectionRound.run(iterationInput)

    if not iteration.wasDecidedByLot():
        title = f"Round {iteration.getRoundNumber()}  (simulated ballot)"
        fig = iteration.plot_vote_share_pie(title=title)
        display(fig)
        plt.close(fig)

if iteration.wasDecidedByLot():
    print(f"The election was decided in round {iteration.getRoundNumber()} by a random procedure.")
    if iteration.getDrawLotsStrategy() == DrawLotsStrategy.RANDOM:
        print(f"A drawing of lots was used as the random procedure.")
    elif iteration.getDrawLotsStrategy() == DrawLotsStrategy.MARGINAL_LEAD:
        print(f"Marginal vote differences, treated as random outcomes, decided the election.")
    else:
        raise ValueError(f"Unknown DrawLotsStrategy: {iteration.getDrawLotsStrategy()}")


print(f"The winner is: {election.getWinner().name}")

In [ ]:
from ipres import VoteMatrixAnalyzer
import pandas as pd

pd.set_option('display.max_rows', 10)
vm_analyzer = VoteMatrixAnalyzer(election.getFirstIteration().vote_matrix.getVotes())
display(vm_analyzer.show_relative_vote_matrix(styler=False))
display(vm_analyzer.show_constituency_importance_matrix(styler=False, decimals=4))

## Display Seat Distribution

In [ ]:
import pandas as pd
from ipres.allocation import ConstituencyAllocationMethod
evaluator = ElectionEvaluator(
    seed=election_config.seed,
    seat_distribution_method=SeatDistributionMethod.SAINTE_LAGUE,
    constituency_allocation_method=ConstituencyAllocationMethod.OPTIMAL,
    quota_correction_strategy=QuotaCorrectionStrategy.FAVOR_LARGE_PARTIES,
)
result = evaluator.evaluate(election)
seat_distribution_table = result.get_seat_distribution_table()
display(seat_distribution_table)

pie = result.plot_seat_share_pie(min_seats_for_display=10)
display(pie)
plt.close(pie)


## 3. Display Constituency Assignments

In [ ]:
# Summary of constituency assignments
summary_table = result.get_constituency_summary_table()
print("Constituency summary by party:")
display(summary_table)

display(result.get_constituency_assignment_table(sort_by="constituency"))